In [ ]:
import os
%cd /content/
%ls
!unzip 4_colab
%cd 4_colab/


!pip install -r requirements.txt



    
%cd 4_colab/
!pip install --no-deps secml_malware
!pip install --no-deps secml
input_folder = './samples/'
output_folder = './output/'
import utils as ut
import os
import subprocess
from tqdm import trange


In [ ]:
def craft_adversarial_sample(image, file_name, len_of_exe, model, iterations, padding_length):
    """
    Iteratively applies FGSM to craft an adversarial sample.
    """
    # Simply clone the tensor to save the "Before" state

    out_path = os.path.join(output_folder, file_name)

    for iteration in trange(iterations, desc=f"Crafting {file_name}", unit="iter"):
        # Apply one FGSM step
        prediction, image = ut.fgsm(model, image, len_of_exe, padding_length)

        # Save updated image back into executable
        ut.img_to_exe(image, file_path, out_path, len_of_exe, padding_length)

        # Check if the attack was successful
        if ut.predict_with_target_model(out_path) == 'Benign':
            return True

    return False

In [ ]:
if __name__ == '__main__':

    padding_length = 2048

    # Load the Surrogate Models
    ResNet50 = ut.load_resnet50()
    MobileNetV2 = ut.load_mobilenetv2()
    for i, file_name in enumerate(os.listdir(input_folder)):
        file_path = os.path.join(input_folder, file_name)
        with open(file_path, "rb") as file_handle:
            code = file_handle.read()

        # Create an image from the EXE file
        img, len_of_exe = ut.exe_to_img(code)
        copy_img = img.clone()

        # complete craft_adversarial_sample method and call the method for both the surrogate models.
        success = craft_adversarial_sample(img, file_name, len_of_exe, ResNet50, 50, padding_length)
        if success:
            print("Adversarial sample creation for the file", file_name, "is complete Resnet50")
            continue

        # Fixed call: include file_name before len_of_exe
        success = craft_adversarial_sample(copy_img, file_name, len_of_exe, MobileNetV2, 50, padding_length
                                           )
        if success:
            print("Adversarial sample creation for the file", file_name, "is complete MobileNet V2")
            continue
        print('failed', file_name)

    print("\n[+] Attack phase complete. Running grading script...")
    subprocess.run(["python", "grading_script.py"], check=True)